# Đào tạo Keyword Spotting với mô hình Honk (Res15)

Notebook này đã được cập nhật toàn bộ các bản vá lỗi tự động để tương thích 100% với môi trường Kaggle hiện tại. Các lỗi liên quan đến `librosa`, `pyaudio`, `pcen` và `chainmap` đều được xử lý ngầm ở Bước 2 & 3.

### Cách chạy:
1. Đảm bảo ở mục **Accelerator** bên phải đã chọn **GPU T4x2** hoặc **GPU P100**.
2. Bấm **Run All** để chạy tuần tự từ trên xuống dưới.

In [ ]:
# Bước 1: Tải mã nguồn dự án Honk
!git clone https://github.com/castorini/honk.git
%cd honk

In [ ]:
# Bước 2: Cập nhật thư viện hệ thống và môi trường
!sed -i 's/==.*//g' requirements.txt
!sed -i 's/>=.*//g' requirements.txt
!sed -i 's/<=.*//g' requirements.txt

!apt-get update > /dev/null
!apt-get install -y portaudio19-dev > /dev/null
!pip install pyaudio git+https://github.com/daemon/pytorch-pcen > /dev/null
!pip install -r requirements.txt > /dev/null
print("Hoàn tất cài đặt thư viện!")

In [ ]:
# Bước 3: Vá các hàm cũ của thư viện Python (Monkey-patch)
with open('utils/model.py', 'r') as f:
    content = f.read()
with open('utils/model.py', 'w') as f:
    f.write(content.replace('from chainmap import ChainMap', 'from collections import ChainMap'))

with open('utils/manage_audio.py', 'r') as f:
    content = f.read()

patch = """
import numpy as np
if not hasattr(librosa.filters, 'dct'):
    def _dct(n_filters, n_input):
        basis = np.empty((n_filters, n_input))
        basis[0, :] = 1.0 / np.sqrt(n_input)
        samples = np.arange(1, 2 * n_input, 2) * np.pi / (2.0 * n_input)
        for i in range(1, n_filters):
            basis[i, :] = np.cos(i * samples) * np.sqrt(2.0 / n_input)
        return basis
    librosa.filters.dct = _dct
"""
with open('utils/manage_audio.py', 'w') as f:
    f.write(content.replace('import librosa\n', 'import librosa\n' + patch))
    
print("Đã vá code cũ thành công!")

In [ ]:
# Bước 4: Tải Google Speech Commands Dataset và các Mô hình Pre-trained
!chmod +x fetch_data.sh
!./fetch_data.sh

In [ ]:
# Bước 5: Bắt đầu quá trình Huấn luyện (End-to-End Training)
# Sử dụng mô hình mạng Residual Network 15 lớp (res15)
!python -m utils.train --data_folder training_data --model res15 --wanted_words yes no up down left right on off stop go --dev_every 1 --n_labels 12 --n_epochs 26 --weight_decay 0.00001 --lr 0.1 0.01 0.001 --schedule 3000 6000